In [1]:
import glob, json, numpy as np, pandas as pd, lightgbm as lgb

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found: check the attached datasets (and the aes_dataset version = v2)"
    return hits[0]

T = ["cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions"]
oof = pd.read_csv(find("oof_ell_deberta.csv"))[["text_id"] + [f"oof_{t}" for t in T]]
ell = (pd.read_csv(find("ell_prepared.csv"))
       .merge(pd.read_csv(find("ell_features.csv")), on="text_id")
       .merge(pd.read_csv(find("ell_languagetool.csv")), on="text_id")
       .merge(pd.read_csv(find("ell_cola.csv")), on="text_id")
       .merge(oof, on="text_id"))
oof_cols = [f"oof_{t}" for t in T]
feature_cols = [c for c in ell.columns if c not in set(T) | {"text_id", "text", "fold", "word_count"} and not c.startswith("oof_")]
Y, fold, D = ell[T].values, ell["fold"].values, ell[oof_cols].values
print(len(feature_cols), "features |", ell.shape)

params = dict(n_estimators=600, learning_rate=0.03, num_leaves=31, min_child_samples=30, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=42, verbose=-1)
def lgb_oof(cols):
    out = np.zeros_like(Y)
    for f in range(5):
        tr, va = fold != f, fold == f
        for j in range(len(T)):
            out[va, j] = lgb.LGBMRegressor(**params).fit(ell.loc[tr, cols], Y[tr, j]).predict(ell.loc[va, cols])
    return out

L = lgb_oof(feature_cols)
S = lgb_oof(feature_cols + oof_cols)
rmse = lambda P: np.sqrt(((P - Y) ** 2).mean(0))
grid = np.linspace(0, 1, 21)
w = np.array([min(grid, key=lambda g: np.sqrt(((g * D[:, j] + (1 - g) * L[:, j] - Y[:, j]) ** 2).mean())) for j in range(len(T))])
B = np.clip(w * D + (1 - w) * L, 1, 5)

table = pd.DataFrame({"DeBERTa": rmse(D), "LightGBM": rmse(L), "Blend": rmse(B), "Stack": rmse(np.clip(S, 1, 5))}, index=T)
table.loc["MEAN"] = table.mean()
print("\nRMSE per trait (lower is better):")
print(table.round(3).to_string())
print("\nBlend weight on DeBERTa per trait:", dict(zip(T, w.round(2))))

pd.DataFrame(np.column_stack([ell["text_id"], Y, D, L, B]),
             columns=["text_id"] + T + [f"deberta_{t}" for t in T] + [f"lgbm_{t}" for t in T] + [f"blend_{t}" for t in T]
             ).to_csv("/kaggle/working/ell_stacking_oof.csv", index=False)
for j, t in enumerate(T):
    lgb.LGBMRegressor(**params).fit(ell[feature_cols], Y[:, j]).booster_.save_model(f"/kaggle/working/lgbm_ell_{t}.txt")
json.dump({"deberta_weights": dict(zip(T, map(float, w))), "feature_cols": feature_cols},
          open("/kaggle/working/ell_blend_config.json", "w"), indent=1)

64 features | (3911, 80)

RMSE per trait (lower is better):
             DeBERTa  LightGBM  Blend  Stack
cohesion       0.496     0.528  0.489  0.509
syntax         0.457     0.479  0.450  0.469
vocabulary     0.421     0.440  0.416  0.436
phraseology    0.467     0.486  0.459  0.480
grammar        0.489     0.503  0.479  0.498
conventions    0.458     0.476  0.448  0.466
MEAN           0.464     0.486  0.457  0.476

Blend weight on DeBERTa per trait: {'cohesion': np.float64(0.7), 'syntax': np.float64(0.65), 'vocabulary': np.float64(0.7), 'phraseology': np.float64(0.65), 'grammar': np.float64(0.6), 'conventions': np.float64(0.65)}
